# H2. NASA Turbofan Engine — RUL(잔여수명) 예측

**잔여수명(Remaining Useful Life, RUL)** 예측은 예지보전(Predictive Maintenance)의 핵심 과제입니다.
엔진이 고장 나기 전까지 얼마나 남았는지를 예측하여, 미리 유지보수를 계획할 수 있습니다.

### 데이터셋: NASA C-MAPSS

**C-MAPSS** (Commercial Modular Aero-Propulsion System Simulation)는 NASA가 공개한 항공엔진 시뮬레이터입니다.

| 항목 | 설명 |
|------|------|
| 하위셋 | FD001 ~ FD004 (운영조건/고장모드 조합) |
| 컬럼 수 | 26개 (unit, cycle, 3개 op_setting, 21개 센서) |
| 학습 데이터 | **Run-to-Failure** — 엔진이 고장 날 때까지의 전체 기록 |
| 테스트 데이터 | 고장 이전 시점까지의 기록 + **실제 RUL 정답** |

### 이 노트북에서 배울 내용

- <span style="color:#27AE60">**RUL 라벨링**</span>: Linear RUL vs Piecewise Clipped RUL, 왜 클리핑이 필요한지
- <span style="color:#E74C3C; font-weight:bold">**상수 센서 탐지**</span>: 표준편차 기반 노이즈 제거
- <span style="color:#2980B9">**Sliding Window**</span>: 시계열을 LSTM 입력 시퀀스로 변환
- <span style="color:#E74C3C; font-weight:bold">**NASA S-Score**</span>: 비대칭 페널티 평가 지표
- <span style="color:#E74C3C; font-weight:bold">**클리핑 효과 비교**</span>: Linear vs Clipped RUL 성능 차이 검증

> **참고 캐글 메달 노트북**: Damir Kamalov "RUL engines", fabf001 "NASA Turbofan Jet Engine LSTM Loss s-score",
> Wassim Derbel "NASA Predictive Maintenance (RUL)", sumitpr96 "AIML Project"


In [1]:
import os
import sys
import time
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.optimizers import Adam

I0000 00:00:1780378126.763209 2470993 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780378126.823434 2470993 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1780378128.257483 2470993 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("/home/growingwithai/dev/205-research-etri/lecture/h2_debug.log", mode="w"),
    ]
)
log = logging.getLogger("NASA-RUL")

---
### 실험 설정

NASA C-MAPSS 예측 파이프라인의 핵심 하이퍼파라미터를 정의합니다.

```{admonition} 왜 RUL_CAP=125인가?
:class: tip

캐글 메달 노트북들에서 가장 널리 쓰이는 클리핑 상한값입니다.
초기 엔진 수명에서는 **열화(degradation) 패턴이 관측되지 않으므로**, RUL이 125 이상이면 모두 125로 처리합니다.
이렇게 하면 <span style="color:#E74C3C; font-weight:bold">모델이 의미 있는 열화 구간에만 집중</span>하게 됩니다.
```
| 파라미터 | 값 | 의미 |
|----------|-----|------|
| <span style="color:#2980B9">DATASET_ID</span> | FD001 | 단일 운영조건 + 단일 고장모드 |
| <span style="color:#2980B9">RUL_CAP</span> | 125 | Piecewise Linear RUL 클리핑 상한 |
| <span style="color:#2980B9">SEQUENCE_LENGTH</span> | 30 | 30사이클의 과거 데이터를 보고 RUL 예측 |
| <span style="color:#2980B9">BATCH_SIZE</span> | 256 | 미니배치 크기 |
| <span style="color:#2980B9">EPOCHS</span> | 50 | 학습 에포크 수 |


In [3]:
DATA_DIR = "/home/growingwithai/dev/205-research-etri/lecture/h2_data_copy/CMaps"
DATASET_ID = "FD001"        # FD001~FD004 중 선택
RUL_CAP = 125               # Piecewise Linear RUL 클리핑 상한
SEQUENCE_LENGTH = 30        # Sliding window 길이
BATCH_SIZE = 256
EPOCHS = 50
LR = 0.001
RANDOM_STATE = 42

# 원본 데이터에는 컬럼명이 없음 (공백으로 구분된 숫자만 있음)
# readme.txt에 명시된 26개 컬럼명을 수동으로 정의하여 부여
COL_NAMES = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

In [4]:
class Timer:
    """with문과 함께 사용하는 컨텍스트 매니저 — 각 단계별 소요시간 측정"""
    def __init__(self, name):
        self.name = name
        self.t0 = time.time()
        log.info(f"[START] {self.name}")

    def __enter__(self):
        return self

    def __exit__(self, *args):
        elapsed = time.time() - self.t0
        log.info(f"[DONE] {self.name} — {elapsed:.2f}s")


def debug_df(df, name="DataFrame"):
    log.info(f"  [{name}] shape={df.shape}, units={df['unit'].nunique()}, "
             f"cycles_range=({df['cycle'].min()},{df['cycle'].max()}), "
             f"nulls={df.isnull().sum().sum()}")

---
### 데이터 로드

NASA C-MAPSS 데이터 파일은 **공백으로 구분된 텍스트 파일**로, 컬럼 헤더가 없습니다.
`sep=r"\s+"`로 하나 이상의 공백을 구분자로 인식하고, 수동 정의한 26개 컬럼명을 부여합니다.

<span style="color:#27AE60">데이터 흐름:</span> `train_FD001.txt` (학습용 run-to-failure) + `test_FD001.txt` (테스트용) + `RUL_FD001.txt` (테스트 엔진의 실제 잔여수명 정답)

| 컬럼 | 개수 | 설명 |
|------|------|------|
| unit, cycle | 2 | 엔진 ID, 운영 사이클 |
| op_setting_1~3 | 3 | 운영 설정값 |
| sensor_1~21 | 21 | 센서 측정값 (온도, 압력, 팬속도 등) |


In [5]:
with Timer("데이터 로드"):
    train_path = os.path.join(DATA_DIR, f"train_{DATASET_ID}.txt")
    test_path  = os.path.join(DATA_DIR, f"test_{DATASET_ID}.txt")
    rul_path   = os.path.join(DATA_DIR, f"RUL_{DATASET_ID}.txt")

    train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=COL_NAMES)
    test_df  = pd.read_csv(test_path,  sep=r"\s+", header=None, names=COL_NAMES)
    rul_true = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["rul_true"])

    debug_df(train_df, "Train")
    debug_df(test_df,  "Test")
    log.info(f"  RUL true: shape={rul_true.shape}, range=({rul_true['rul_true'].min()},{rul_true['rul_true'].max()})")

[14:28:48] INFO — [START] 데이터 로드


[14:28:48] INFO —   [Train] shape=(4163, 26), units=100, cycles_range=(1,361), nulls=0


[14:28:48] INFO —   [Test] shape=(2663, 26), units=100, cycles_range=(1,301), nulls=0


[14:28:48] INFO —   RUL true: shape=(100, 1), range=(7,145)


[14:28:48] INFO — [DONE] 데이터 로드 — 0.04s


---
### EDA — 센서 분석 및 상수 센서 탐지

21개 센서 중 **정보량이 없는 상수 센서**를 탐지합니다.
이 단계는 전체 파이프라인에서 매우 중요합니다.

```{admonition} 핵심 개념 — 상수 센서 제거
:class: tip

<span style="color:#E74C3C; font-weight:bold">표준편차 < 0.5</span>인 센서는 값이 거의 변하지 않아 **노이즈**로 작용합니다.
예: sensor_1(항상 518.67), sensor_5(항상 14.62) — 이런 센서를 그대로 사용하면 모델 성능이 저하됩니다.
캐글 메달 노트북들의 **공통 전처리 단계**입니다.
```
- <span style="color:#E74C3C; font-weight:bold">탐지 기준</span>: 센서별 표준편차 계산 → `std < 0.5`이면 상수 센서로 분류
- <span style="color:#27AE60">결과</span>: 상수 센서는 제거하고, 유효 센서만 특징으로 사용
- 센서 트렌드 시각화에서 **빨간 배경** = 상수 센서


In [6]:
with Timer("EDA"):
    sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
    sensor_std = train_df[sensor_cols].std().sort_values()

    log.info("  센서별 표준편차 (낮은 순):")
    for col, std in sensor_std.items():
        flag = " *** 상수/거의 상수" if std < 0.5 else ""
        log.info(f"    {col}: std={std:.4f}{flag}")

    constant_sensors = sensor_std[sensor_std < 0.5].index.tolist()
    useful_sensors   = [s for s in sensor_cols if s not in constant_sensors]
    log.info(f"  제거 대상 상수 센서 ({len(constant_sensors)}개): {constant_sensors}")
    log.info(f"  유효 센서 ({len(useful_sensors)}개): {useful_sensors}")

[14:28:48] INFO — [START] EDA


[14:28:48] INFO —   센서별 표준편차 (낮은 순):


[14:28:48] INFO —     sensor_1: std=0.0000 *** 상수/거의 상수


[14:28:48] INFO —     sensor_18: std=0.0000 *** 상수/거의 상수


[14:28:48] INFO —     sensor_19: std=0.0000 *** 상수/거의 상수


...

[14:28:48] INFO —     sensor_14: std=18.8885


[14:28:48] INFO —     sensor_9: std=21.8846


[14:28:48] INFO —   제거 대상 상수 센서 (14개): ['sensor_1', 'sensor_18', 'sensor_19', 'sensor_16', 'sensor_10', 'sensor_5', 'sensor_6', 'sensor_15', 'sensor_8', 'sensor_13', 'sensor_21', 'sensor_20', 'sensor_11', 'sensor_2']


[14:28:48] INFO —   유효 센서 (7개): ['sensor_3', 'sensor_4', 'sensor_7', 'sensor_9', 'sensor_12', 'sensor_14', 'sensor_17']


[14:28:48] INFO — [DONE] EDA — 0.03s


In [7]:
    # 센서 트렌드 시각화 (샘플 엔진)
    sample_unit = train_df["unit"].iloc[0]
    sample_df   = train_df[train_df["unit"] == sample_unit]

    fig, axes = plt.subplots(nrows=7, ncols=3, figsize=(18, 20))
    for idx, sensor in enumerate(sensor_cols):
        ax = axes[idx // 3, idx % 3]
        ax.plot(sample_df["cycle"], sample_df[sensor], linewidth=0.8)
        ax.set_title(sensor, fontsize=9)
        if sensor in constant_sensors:
            ax.set_facecolor("#ffcccc")
    fig.suptitle(f"센서 트렌드 — Unit {sample_unit} (빨간 배경=상수 센서)", fontsize=14)
    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_sensor_trends.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_sensor_trends.png")

/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 49468 (\N{HANGUL SYLLABLE SEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 49436 (\N{HANGUL SYLLABLE SEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 53944 (\N{HANGUL SYLLABLE TEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 47116 (\N{HANGUL SYLLABLE REN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 46300 (\N{HANGUL SYLLABLE DEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 48744 (\N{HANGUL SYLLABLE BBAL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/609699246.py:13: UserWarning: Glyph 44036 (\N{HANGUL SYLLABLE GAN}) missing from font(s) DejaVu Sans

/tmp/ipykernel_2470993/609699246.py:14: UserWarning: Glyph 49468 (\N{HANGUL SYLLABLE SEN}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_sensor_trends.png", dpi=150)
/tmp/ipykernel_2470993/609699246.py:14: UserWarning: Glyph 49436 (\N{HANGUL SYLLABLE SEO}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_sensor_trends.png", dpi=150)
/tmp/ipykernel_2470993/609699246.py:14: UserWarning: Glyph 53944 (\N{HANGUL SYLLABLE TEU}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_sensor_trends.png", dpi=150)
/tmp/ipykernel_2470993/609699246.py:14: UserWarning: Glyph 47116 (\N{HANGUL SYLLABLE REN}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_sensor_trends.png", dpi=150)
/tmp/ipykernel_2470993/609699246.py:14: UserWarning: Glyph 46300 (\N{HANGUL SYLLABLE DEU}) missing from font(s) 

[14:28:51] INFO —   → 저장: lecture/h2_sensor_trends.png


---
### RUL 라벨 생성 — Linear vs Piecewise Clipped

엔진의 잔여수명(RUL)을 라벨로 생성합니다. 두 가지 방식을 비교합니다.

| 방식 | 공식 | 특징 |
|------|------|------|
| **Linear RUL** | `max_cycle - current_cycle` | 단순 카운트다운 |
| **Clipped RUL** | `min(linear_RUL, cap)` | 상한선 적용 |

```{admonition} 왜 클리핑이 필요한가?
:class: tip

초기 엔진 수명에서는 **열화(degradation) 패턴이 관측되지 않습니다.**
<span style="color:#E74C3C; font-weight:bold">cap=125</span>로 자르면 모델이 의미 있는 열화 구간에만 집중합니다.

예시 (엔진 최대 수명 = 193사이클):
- cycle 1: linear_RUL=192, <span style="color:#E74C3C; font-weight:bold">clipped_RUL=125</span>
- cycle 68: linear_RUL=125, clipped_RUL=125 ← 여기서부터 감소 시작
- cycle 193: linear_RUL=0, clipped_RUL=0
```
<span style="color:#27AE60">데이터 변환:</span>
1. 각 엔진의 마지막 사이클(`max_cycle`)을 구함
2. Linear RUL = `max_cycle - 현재 cycle`
3. Clipped RUL = `.clip(upper=cap)`로 상한선 적용


In [8]:
def add_rul_label(df, cap=None):
    """
    각 엔진(유닛)의 각 시점(cycle)에 대해 RUL 라벨을 생성.
    - Linear RUL: max_cycle - current_cycle
    - Clipped RUL (cap): min(linear_rul, cap)
    """
    result = df.copy()
    max_cycles = result.groupby("unit")["cycle"].max().reset_index()
    max_cycles.columns = ["unit", "max_cycle"]
    result = result.merge(max_cycles, on="unit")

    result["rul_linear"] = result["max_cycle"] - result["cycle"]

    if cap is not None:
        result[f"rul_clip_{cap}"] = result["rul_linear"].clip(upper=cap)
        log.info(f"  RUL 클리핑 적용: cap={cap} "
                 f"(linear range: [{result['rul_linear'].min()}, {result['rul_linear'].max()}] "
                 f"→ clipped range: [{result[f'rul_clip_{cap}'].min()}, {result[f'rul_clip_{cap}'].max()}])")
    return result

In [9]:
with Timer("RUL 라벨 생성"):
    train_df = add_rul_label(train_df, cap=RUL_CAP)
    debug_df(train_df, "Train (RUL added)")

[14:28:51] INFO — [START] RUL 라벨 생성


[14:28:51] INFO —   RUL 클리핑 적용: cap=125 (linear range: [0, 360] → clipped range: [0, 125])


[14:28:51] INFO —   [Train (RUL added)] shape=(4163, 29), units=100, cycles_range=(1,361), nulls=0


[14:28:51] INFO — [DONE] RUL 라벨 생성 — 0.01s


In [10]:
    # RUL 분포 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(train_df["rul_linear"], bins=50, color="#2196F3", alpha=0.7)
    axes[0].set_title("Linear RUL 분포")
    axes[0].set_xlabel("RUL (cycles)")
    axes[0].axvline(x=RUL_CAP, color="red", linestyle="--", label=f"cap={RUL_CAP}")
    axes[0].legend()

    axes[1].hist(train_df[f"rul_clip_{RUL_CAP}"], bins=50, color="#4CAF50", alpha=0.7)
    axes[1].set_title(f"Clipped RUL 분포 (cap={RUL_CAP})")
    axes[1].set_xlabel("RUL (cycles)")

    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_distribution.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_rul_distribution.png")

/tmp/ipykernel_2470993/1851481611.py:13: UserWarning: Glyph 48516 (\N{HANGUL SYLLABLE BUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1851481611.py:13: UserWarning: Glyph 54252 (\N{HANGUL SYLLABLE PO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1851481611.py:14: UserWarning: Glyph 48516 (\N{HANGUL SYLLABLE BUN}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_distribution.png", dpi=150)
/tmp/ipykernel_2470993/1851481611.py:14: UserWarning: Glyph 54252 (\N{HANGUL SYLLABLE PO}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_distribution.png", dpi=150)


[14:28:51] INFO —   → 저장: lecture/h2_rul_distribution.png


In [11]:
    # 샘플 엔진 RUL 비교
    fig, ax = plt.subplots(figsize=(12, 4))
    sample_rul = train_df[train_df["unit"] == sample_unit]
    ax.plot(sample_rul["cycle"], sample_rul["rul_linear"], label="Linear RUL", linewidth=2)
    ax.plot(sample_rul["cycle"], sample_rul[f"rul_clip_{RUL_CAP}"], label=f"Clipped RUL (cap={RUL_CAP})",
            linewidth=2, linestyle="--", color="red")
    ax.set_xlabel("Cycle")
    ax.set_ylabel("RUL")
    ax.set_title(f"RUL 비교 — Unit {sample_unit}")
    ax.legend()
    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_comparison.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_rul_comparison.png")

[14:28:52] INFO —   → 저장: lecture/h2_rul_comparison.png


/tmp/ipykernel_2470993/2058479933.py:11: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/2058479933.py:11: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/2058479933.py:12: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_comparison.png", dpi=150)
/tmp/ipykernel_2470993/2058479933.py:12: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_comparison.png", dpi=150)


---
### 전처리 — 상수 센서 제거 & MinMax 정규화

EDA에서 탐지한 상수 센서를 제거하고, 유효 센서에 대해 **MinMaxScaler**를 적용합니다.

```{admonition} 핵심 — 데이터 누수(Data Leakage) 방지
:class: tip

<span style="color:#E74C3C; font-weight:bold">`fit_transform`은 학습 데이터에만 적용</span>하고,
테스트 데이터에는 <span style="color:#2980B9">`transform`만</span> 적용합니다.
테스트 데이터로 fit하면 모델이 테스트 정보를 미리 알게 되어 평가가 부정확해집니다.
```
<span style="color:#27AE60">변환 흐름:</span> 상수 센서 6~7개 제거 → 14~15개 유효 센서에 대해 `[0, 1]` 범위 정규화


In [12]:
with Timer("전처리"):
    feature_cols = useful_sensors  # 상수 센서 제거된 센서만 사용

    scaler = MinMaxScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

    log.info(f"  정규화 완료: {len(feature_cols)}개 센서 (상수 {len(constant_sensors)}개 제거)")
    log.info(f"  Train 센서 범위: [{train_df[feature_cols].min().min():.4f}, {train_df[feature_cols].max().max():.4f}]")
    log.info(f"  Test  센서 범위: [{test_df[feature_cols].min().min():.4f}, {test_df[feature_cols].max().max():.4f}]")

[14:28:52] INFO — [START] 전처리


[14:28:52] INFO —   정규화 완료: 7개 센서 (상수 14개 제거)


[14:28:52] INFO —   Train 센서 범위: [0.0000, 1.0000]


[14:28:52] INFO —   Test  센서 범위: [-0.0435, 1.0135]


[14:28:52] INFO — [DONE] 전처리 — 0.02s


---
### Sliding Window 시퀀스 생성

시계열 RUL 예측에서 **sliding window**는 LSTM에 과거 N개 사이클을 시퀀스로 입력하기 위한 핵심 기법입니다.

```{admonition} 핵심 개념 — Sliding Window
:class: tip

- <span style="color:#E74C3C; font-weight:bold">stride=1</span> (한 칸씩 이동) → **데이터 증강 효과**: 길이 L인 엔진에서 `(L - seq_length + 1)`개의 샘플이 생성됨
- <span style="color:#2980B9">타겟</span>은 시퀀스의 **마지막 시점**의 RUL (`i + seq_length - 1`)
- 길이가 `seq_length` 미만인 엔진은 건너뜀 (정보 부족)
```
<span style="color:#27AE60">데이터 변환:</span>
`DataFrame (N, features)` → `3D array (samples, SEQUENCE_LENGTH, features)`


In [13]:
def generate_sequences(df, feature_cols, target_col, seq_length):
    """각 엔진별로 sliding window 시퀀스 생성."""
    X, y, units = [], [], []
    for unit_id, group in df.groupby("unit"):
        data  = group[feature_cols].values
        label = group[target_col].values
        if len(data) < seq_length:
            log.warning(f"  Unit {unit_id}: 시퀀스 길이 {len(data)} < {seq_length}, 건너뜀")
            continue
        for i in range(len(data) - seq_length + 1):
            X.append(data[i : i + seq_length])
            y.append(label[i + seq_length - 1])  # 시퀀스 마지막 시점의 RUL
            units.append(unit_id)
    return np.array(X), np.array(y), np.array(units)

In [14]:
with Timer("Sliding Window 시퀀스 생성"):
    target_col = f"rul_clip_{RUL_CAP}"

    X_train, y_train, _ = generate_sequences(train_df, feature_cols, target_col, SEQUENCE_LENGTH)
    log.info(f"  Train 시퀀스: X={X_train.shape}, y={y_train.shape}")
    log.info(f"  y_train 통계: mean={y_train.mean():.1f}, std={y_train.std():.1f}, "
             f"range=[{y_train.min()}, {y_train.max()}]")

[14:28:52] INFO — [START] Sliding Window 시퀀스 생성


[14:28:52] WARNING —   Unit 39: 시퀀스 길이 26 < 30, 건너뜀


[14:28:52] WARNING —   Unit 57: 시퀀스 길이 28 < 30, 건너뜀


[14:28:52] WARNING —   Unit 70: 시퀀스 길이 28 < 30, 건너뜀


[14:28:52] WARNING —   Unit 91: 시퀀스 길이 27 < 30, 건너뜀


[14:28:52] INFO —   Train 시퀀스: X=(1270, 30, 7), y=(1270,)


[14:28:52] INFO —   y_train 통계: mean=44.1, std=36.3, range=[0, 125]


[14:28:52] INFO — [DONE] Sliding Window 시퀀스 생성 — 0.08s


---
### 테스트 데이터 준비

학습과 테스트의 **시퀀스 생성 방식이 다릅니다**.

| | 학습 데이터 | 테스트 데이터 |
|---|---|---|
| 시퀀스 | 모든 sliding window 생성 | 각 엔진의 **마지막 시퀀스 1개만** |
| 이유 | 데이터 증강 | Kaggle 평가: 마지막 시점의 RUL 예측 |

```{admonition} 패딩 처리
:class: tip

시퀀스 길이보다 짧은 엔진은 <span style="color:#2980B9">앞을 0으로 패딩</span>합니다.
실제 데이터를 시퀀스 **뒤쪽에 배치**하여 최근 데이터가 LSTM에 우선 전달되도록 합니다.
```

In [15]:
def get_test_sequences(df, feature_cols, seq_length):
    """각 엔진의 마지막 시퀀스만 추출 (최종 시점 RUL 예측용)."""
    X_test, units = [], []
    for unit_id, group in df.groupby("unit"):
        data = group[feature_cols].values
        if len(data) < seq_length:
            padded = np.zeros((seq_length, len(feature_cols)))
            padded[-len(data):] = data
            X_test.append(padded)
        else:
            X_test.append(data[-seq_length:])
        units.append(unit_id)
    return np.array(X_test), np.array(units)

In [16]:
with Timer("테스트 데이터 준비"):
    X_test_final, test_units = get_test_sequences(test_df, feature_cols, SEQUENCE_LENGTH)
    y_test_final = rul_true["rul_true"].values

    log.info(f"  Test 시퀀스: X={X_test_final.shape}, y_true={y_test_final.shape}")
    log.info(f"  y_test 통계: mean={y_test_final.mean():.1f}, std={y_test_final.std():.1f}, "
             f"range=[{y_test_final.min()}, {y_test_final.max()}]")

[14:28:52] INFO — [START] 테스트 데이터 준비


[14:28:52] INFO —   Test 시퀀스: X=(100, 30, 7), y_true=(100,)


[14:28:52] INFO —   y_test 통계: mean=75.5, std=41.6, range=[7, 145]


[14:28:52] INFO — [DONE] 테스트 데이터 준비 — 0.05s


---
### LSTM 모델 아키텍처 및 학습

#### 모델 구조

| 레이어 | 설정 | 역할 |
|--------|------|------|
| <span style="color:#E74C3C; font-weight:bold">LSTM(64)</span> | `return_sequences=True` | 모든 시간단계 출력 → 다음 LSTM에 전달 |
| <span style="color:#E74C3C; font-weight:bold">LSTM(32)</span> | `return_sequences=False` | 마지막 시간단계만 출력 → 시퀀스 요약 |
| Dense(64→32) | ReLU | 비선형 특징 추출 |
| Dense(1) | <span style="color:#E74C3C; font-weight:bold">`linear` 활성화</span> | RUL 회귀값 그대로 출력 (양수/음수 제한 없음) |
| BatchNormalization | — | 학습 안정화, 기울기 폭발/소실 방지 |

#### NASA S-Score — 비대칭 평가 지표

```{admonition} 핵심 개념 — 왜 늦은 예측이 더 위험한가?
:class: tip

`d = 예측 - 실제` (오차의 **부호**가 중요합니다)

| 상황 | 조건 | 페널티 | 의미 |
|------|------|--------|------|
| 조기 예측 | d < 0 | `exp(-d/10) - 1` | 덜 벌점 (일찍 경고는 괜찮음) |
| <span style="color:#E74C3C; font-weight:bold">늦은 예측</span> | d >= 0 | `exp(d/13) - 1` | **더 벌점** (고장 후에 경고 = 위험) |

<span style="color:#E74C3C; font-weight:bold">항공엔진에서 늦은 고장 예측은 치명적</span>이므로, 비대칭 페널티를 통해 안전 측의 예측을 유도합니다.
```

In [17]:
class RealtimeDebugCallback(Callback):
    """매 에포크마다 상세 로그 출력."""
    def __init__(self, log_every=5):
        super().__init__()
        self.log_every = log_every

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if (epoch + 1) % self.log_every == 0 or epoch == 0:
            log.info(
                f"  Epoch {epoch+1:03d}/{EPOCHS} — "
                f"loss={logs.get('loss', 0):.4f} "
                f"mae={logs.get('mae', 0):.4f} "
                f"val_loss={logs.get('val_loss', 0):.4f} "
                f"val_mae={logs.get('val_mae', 0):.4f} "
                f"lr={float(self.model.optimizer.learning_rate):.6f}"
            )

In [18]:
def nasa_s_score(y_true, y_pred):
    """
    NASA S-Score: s = exp(-d/10) - 1    if d < 0 (조기 예측)
                   = exp(d/13) - 1       if d >= 0 (늦은 예측)
    d = y_pred - y_true
    """
    y_true = tf.cast(y_true, tf.float32)
    d = y_pred - y_true
    s = tf.where(d < 0,
                 tf.exp(-d / 10.0) - 1.0,
                 tf.exp(d / 13.0) - 1.0)
    return tf.reduce_mean(s)

In [19]:
with Timer("LSTM 모델 학습"):
    n_features = len(feature_cols)
    model = Sequential([
        LSTM(64, input_shape=(SEQUENCE_LENGTH, n_features), return_sequences=True),
        Dropout(0.2),
        BatchNormalization(),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1, activation="linear")  # 회귀: RUL 값 그대로 출력
    ])

    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss="mse",
        metrics=["mae", nasa_s_score]
    )

    log.info(f"  모델 파라미터 수: {model.count_params():,}")
    model.summary(print_fn=lambda x: log.info(f"  {x}"))

[14:28:52] INFO — [START] LSTM 모델 학습


[14:28:52] INFO —   모델 파라미터 수: 35,457


E0000 00:00:1780378132.262036 2470993 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/growingwithai/dev/205-research-etri/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[14:28:52] INFO —   Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ 

[14:28:52] INFO — [DONE] LSTM 모델 학습 — 0.15s


In [20]:
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[RealtimeDebugCallback(log_every=5)],
        verbose=0
    )

[14:28:57] INFO —   Epoch 001/50 — loss=2794.7383 mae=40.5665 val_loss=4771.3403 val_mae=55.0246 lr=0.001000


[14:28:58] INFO —   Epoch 005/50 — loss=2401.4836 mae=37.4343 val_loss=4588.1343 val_mae=53.7927 lr=0.001000


[14:28:59] INFO —   Epoch 010/50 — loss=1473.0641 mae=29.7274 val_loss=3331.0137 val_mae=43.6753 lr=0.001000


[14:29:01] INFO —   Epoch 015/50 — loss=437.0636 mae=16.0985 val_loss=1497.7465 val_mae=30.5930 lr=0.001000


[14:29:03] INFO —   Epoch 020/50 — loss=270.7289 mae=12.8578 val_loss=1281.9713 val_mae=29.5177 lr=0.001000


...

[14:29:06] INFO —   Epoch 030/50 — loss=180.0374 mae=10.4107 val_loss=1723.7623 val_mae=33.1066 lr=0.001000


[14:29:08] INFO —   Epoch 035/50 — loss=160.2683 mae=9.6571 val_loss=1867.0117 val_mae=34.2792 lr=0.001000


[14:29:09] INFO —   Epoch 040/50 — loss=160.7440 mae=9.4762 val_loss=1549.0934 val_mae=31.5252 lr=0.001000


[14:29:11] INFO —   Epoch 045/50 — loss=138.3835 mae=8.8584 val_loss=1712.4354 val_mae=32.2471 lr=0.001000


[14:29:13] INFO —   Epoch 050/50 — loss=132.5578 mae=8.6297 val_loss=1492.5426 val_mae=30.3240 lr=0.001000


In [21]:
    # 학습 곡선 시각화
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, metric, title in [
        (axes[0], "loss", "Loss (MSE)"),
        (axes[1], "mae", "MAE"),
        (axes[2], "nasa_s_score", "NASA S-Score"),
    ]:
        ax.plot(history.history[metric], label="Train")
        ax.plot(history.history[f"val_{metric}"], label="Val")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_training_curves.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_training_curves.png")

[14:29:13] INFO —   → 저장: lecture/h2_training_curves.png


---
### 평가 — RMSE, S-Score, 시각화

모델의 예측 성능을 **RMSE**와 **NASA S-Score** 두 가지 지표로 평가합니다.

- **RMSE**: 예측 오차의 표준편차 — 값이 작을수록 좋음
- **S-Score**: 비대칭 페널티 — 늦은 예측에 더 큰 벌점

**산점도 해석**: 점이 빨간 점선(Perfect)에 가까울수록 예측이 정확합니다.


In [22]:
with Timer("평가"):
    y_pred = model.predict(X_test_final, verbose=0).flatten()
    y_pred = np.clip(y_pred, 0, None)  # 음수 RUL 방지

    # RMSE
    rmse = np.sqrt(mean_squared_error(y_test_final, y_pred))

    # NASA S-Score (numpy)
    d = y_pred - y_test_final
    s_score = np.mean(np.where(d < 0, np.exp(-d / 10.0) - 1.0, np.exp(d / 13.0) - 1.0))

    log.info(f"  RMSE: {rmse:.2f}")
    log.info(f"  NASA S-Score: {s_score:.4f}")

[14:29:13] INFO — [START] 평가


[14:29:14] INFO —   RMSE: 57.97


[14:29:14] INFO —   NASA S-Score: 31829.8582


[14:29:14] INFO — [DONE] 평가 — 0.61s


In [23]:
    # 예측 vs 실제 산점도 + 엔진별 RUL 비교 바차트
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].scatter(y_test_final, y_pred, alpha=0.5, s=20)
    axes[0].plot([0, max(y_test_final)], [0, max(y_test_final)], "r--", label="Perfect")
    axes[0].set_xlabel("True RUL")
    axes[0].set_ylabel("Predicted RUL")
    axes[0].set_title(f"RUL 예측 vs 실제 (RMSE={rmse:.2f}, S-Score={s_score:.4f})")
    axes[0].legend()

    n_show = min(30, len(y_test_final))
    x_pos = np.arange(n_show)
    width = 0.35
    axes[1].bar(x_pos - width/2, y_test_final[:n_show], width, label="True RUL", alpha=0.7)
    axes[1].bar(x_pos + width/2, y_pred[:n_show], width, label="Predicted RUL", alpha=0.7)
    axes[1].set_xlabel("Unit Index")
    axes[1].set_ylabel("RUL")
    axes[1].set_title(f"엔진별 RUL (처음 {n_show}개)")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_rul_prediction.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_rul_prediction.png")

/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 49892 (\N{HANGUL SYLLABLE SIL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 51228 (\N{HANGUL SYLLABLE JE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 50644 (\N{HANGUL SYLLABLE EN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/637434332.py:21: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from font(s) DejaVu Sans.

[14:29:14] INFO —   → 저장: lecture/h2_rul_prediction.png


In [24]:
    # 오차 분포
    errors = y_pred - y_test_final
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(errors, bins=30, color="#FF9800", alpha=0.7, edgecolor="black")
    axes[0].axvline(x=0, color="red", linestyle="--")
    axes[0].set_title("예측 오차 분포 (Pred - True)")
    axes[0].set_xlabel("Error (cycles)")

    axes[1].scatter(y_test_final, np.abs(errors), alpha=0.5, s=20)
    axes[1].set_xlabel("True RUL")
    axes[1].set_ylabel("|Error|")
    axes[1].set_title("오차 크기 vs True RUL")

    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_error_analysis.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_error_analysis.png")

/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 52264 (\N{HANGUL SYLLABLE CA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 48516 (\N{HANGUL SYLLABLE BUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 54252 (\N{HANGUL SYLLABLE PO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2470993/1163245402.py:14: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sa

[14:29:15] INFO —   → 저장: lecture/h2_error_analysis.png


---
### 클리핑 효과 비교 — Linear vs Clipped RUL

이 실험은 **이 노트북의 핵심 시연 포인트**입니다.
동일한 LSTM 모델을 두 가지 타겟으로 학습하여 성능을 비교합니다.

| 타겟 | RUL 패턴 | 예상 결과 |
|------|----------|-----------|
| **Linear RUL** | 192, 191, ..., 1, 0 | 초기 고RUL 노이즈에 방해받음 |
| <span style="color:#E74C3C; font-weight:bold">**Clipped RUL (cap=125)**</span> | 125, 125, ..., 1, 0 | 의미 있는 열화 구간에만 집중 |

```{admonition} 왜 Clipped RUL이 더 좋은가?
:class: tip

초기 엔진 수명(RUL > 125)에서는 센서 값이 정상 범위에 있어 **열화 신호가 없습니다.**
Linear RUL은 이 구간에서도 192, 191 등 큰 값으로 학습해야 하므로 <span style="color:#E74C3C; font-weight:bold">모델이 불필요한 패턴을 학습</span>하게 됩니다.
Clipped RUL은 이 구간을 모두 125로 만들어 <span style="color:#27AE60">모델이 실제 열화가 시작되는 125 이하 구간에만 집중</span>할 수 있게 합니다.
```

In [25]:
with Timer("클리핑 효과 비교"):
    # Linear RUL로 동일 모델 학습 (비교용)
    target_linear = "rul_linear"
    X_train_lin, y_train_lin, _ = generate_sequences(
        train_df, feature_cols, target_linear, SEQUENCE_LENGTH
    )

    model_linear = Sequential([
        LSTM(64, input_shape=(SEQUENCE_LENGTH, n_features), return_sequences=True),
        Dropout(0.2),
        BatchNormalization(),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1, activation="linear")
    ])
    model_linear.compile(optimizer=Adam(learning_rate=LR), loss="mse", metrics=["mae"])
    model_linear.fit(X_train_lin, y_train_lin, epochs=EPOCHS, batch_size=BATCH_SIZE,
                     validation_split=0.2, verbose=0)

    y_pred_lin = model_linear.predict(X_test_final, verbose=0).flatten()
    y_pred_lin = np.clip(y_pred_lin, 0, None)
    rmse_lin   = np.sqrt(mean_squared_error(y_test_final, y_pred_lin))
    d_lin      = y_pred_lin - y_test_final
    s_score_lin = np.mean(np.where(d_lin < 0, np.exp(-d_lin/10.0)-1.0, np.exp(d_lin/13.0)-1.0))

    log.info(f"  [Linear RUL]    RMSE: {rmse_lin:.2f}, S-Score: {s_score_lin:.4f}")
    log.info(f"  [Clipped {RUL_CAP}]  RMSE: {rmse:.2f}, S-Score: {s_score:.4f}")

[14:29:15] INFO — [START] 클리핑 효과 비교


[14:29:15] WARNING —   Unit 39: 시퀀스 길이 26 < 30, 건너뜀


[14:29:15] WARNING —   Unit 57: 시퀀스 길이 28 < 30, 건너뜀


[14:29:15] WARNING —   Unit 70: 시퀀스 길이 28 < 30, 건너뜀


[14:29:15] WARNING —   Unit 91: 시퀀스 길이 27 < 30, 건너뜀


/home/growingwithai/dev/205-research-etri/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[14:29:36] INFO —   [Linear RUL]    RMSE: 45.75, S-Score: 975.3229


[14:29:36] INFO —   [Clipped 125]  RMSE: 57.97, S-Score: 31829.8582


[14:29:36] INFO — [DONE] 클리핑 효과 비교 — 21.65s


In [26]:
    # 비교 시각화
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, pred, title, rmse_val, s_val in [
        (axes[0], y_pred_lin, f"Linear RUL (RMSE={rmse_lin:.2f})", rmse_lin, s_score_lin),
        (axes[1], y_pred,    f"Clipped RUL cap={RUL_CAP} (RMSE={rmse:.2f})", rmse, s_score),
    ]:
        ax.scatter(y_test_final, pred, alpha=0.5, s=20)
        ax.plot([0, max(y_test_final)], [0, max(y_test_final)], "r--", label="Perfect")
        ax.set_xlabel("True RUL")
        ax.set_ylabel("Predicted RUL")
        ax.set_title(title)
        ax.legend()
    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h2_clipping_comparison.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h2_clipping_comparison.png")

[14:29:37] INFO —   → 저장: lecture/h2_clipping_comparison.png


---
### 결과 요약


In [27]:
SAVE_DIR = "/home/growingwithai/dev/205-research-etri/lecture"
print(f"데이터셋: NASA C-MAPSS {DATASET_ID}")
print(f"사용 센서: {len(feature_cols)}개 (상수 {len(constant_sensors)}개 제거)")
print(f"RUL 클리핑: cap={RUL_CAP}")
print(f"시퀀스 길이: {SEQUENCE_LENGTH}")
print(f"학습 샘플: {X_train.shape[0]}")
print(f"테스트 엔진: {X_test_final.shape[0]}")
print(f"")
print(f"Linear RUL    → RMSE: {rmse_lin:.2f}, S-Score: {s_score_lin:.4f}")
print(f"Clipped({RUL_CAP}) RUL → RMSE: {rmse:.2f}, S-Score: {s_score:.4f}")
print(f"")
print(f"생성된 파일:")
for f in sorted(
    x for x in os.listdir(SAVE_DIR)
    if x.startswith("h2_") and (x.endswith(".png") or x.endswith(".log"))
):
    print(f"    lecture/{f}")
print("디버그 로그: lecture/h2_debug.log")

데이터셋: NASA C-MAPSS FD001
사용 센서: 7개 (상수 14개 제거)
RUL 클리핑: cap=125
시퀀스 길이: 30
학습 샘플: 1270
테스트 엔진: 100

Linear RUL    → RMSE: 45.75, S-Score: 975.3229
Clipped(125) RUL → RMSE: 57.97, S-Score: 31829.8582

생성된 파일:
    lecture/h2_clipping_comparison.png
    lecture/h2_debug.log
    lecture/h2_error_analysis.png
    lecture/h2_rul_comparison.png
    lecture/h2_rul_distribution.png
    lecture/h2_rul_prediction.png
    lecture/h2_sensor_trends.png
    lecture/h2_training_curves.png
디버그 로그: lecture/h2_debug.log
